# SalesMesh synthetic onboarding dataset generator

Run this notebook from top to bottom to generate reproducible synthetic banking, telecom, or mixed onboarding events. It uses Python's standard library only; no Colab setup or package installation is required.

By default the CSV is saved as `data/onboarding_dataset.csv`. In Colab, download it with the final cell and place it in the project's `data` folder if you want to use it with the local model demo. The output includes two independent, group-safe splits:
- `split`: session-level train/validation/test assignment, stratified by industry and scenario.
- `agent_split`: agent-level train/validation/test assignment, keeping all of an agent's sessions in one split.

The generated records are synthetic and must not be represented as real customer data or real-world model evidence. This notebook is the sole source of the dataset generator; there is no separate generator Python script. It runs in local Jupyter or Google Colab without installing third-party packages.


## 1. Choose generation settings
Edit these values before running the generation cell.

In [ ]:
from pathlib import Path

NUMBER_OF_SESSIONS = 6000
INDUSTRY = "both"  # "banking", "telecom", or "both"
SEED = 42
OUTPUT_CSV = Path("data/onboarding_dataset.csv")


## 2. Generator implementation
The following cells are the complete generator. Make generator changes here; there is no separate generator Python script.

In [ ]:
import csv

import random

from collections import defaultdict

from datetime import datetime, timedelta

from pathlib import Path

from typing import Optional

BANKING_FIELDS = [
    {"field": "full_name",                          "stage": "customer_information"},
    {"field": "email_address",                       "stage": "customer_information"},
    {"field": "contact_phone_number",              "stage": "customer_information"},
    {"field": "date_of_birth",                        "stage": "customer_information"},
    {"field": "physical_address",                    "stage": "customer_information"},
    {"field": "identification_document_type",         "stage": "identity_verification"},
    {"field": "identification_document_number",       "stage": "identity_verification"},
    {"field": "selfie_liveness_check",                "stage": "identity_verification"},
    {"field": "face_match_verification",              "stage": "identity_verification"},
    {"field": "profession",                           "stage": "financial_profile"},
    {"field": "income",                               "stage": "financial_profile"},
    {"field": "assets",                               "stage": "financial_profile"},
    {"field": "us_person_status",                     "stage": "regulatory_declarations"},
    {"field": "politically_exposed_person_status",    "stage": "regulatory_declarations"},
    {"field": "terms_and_conditions_acceptance",      "stage": "account_setup"},
    {"field": "commercial_address",                   "stage": "account_setup",
     "condition": lambda ctx: ctx["customer_type"] == "business"},
    {"field": "beneficial_owner_information",         "stage": "account_setup",
     "condition": lambda ctx: ctx["customer_type"] == "business"},
    {"field": "business_relationship_purpose",        "stage": "account_setup",
     "condition": lambda ctx: ctx["customer_type"] == "business"},
    {"field": "business_relationship_nature",         "stage": "account_setup",
     "condition": lambda ctx: ctx["customer_type"] == "business"},
    {"field": "ownership_and_control_information",    "stage": "account_setup",
     "condition": lambda ctx: ctx["customer_type"] == "business"},
]

BANKING_STAGE_ORDER = [
    "customer_information",
    "identity_verification",
    "financial_profile",
    "regulatory_declarations",
    "account_setup",
]

TELECOM_FIELDS = [
    {"field": "full_name",                    "stage": "kyc"},
    {"field": "email_address",                 "stage": "kyc"},
    {"field": "mobile_phone_number",           "stage": "kyc",
     "condition": lambda ctx: ctx["service_type"] == "mobile"},
    {"field": "contact_phone_number",          "stage": "kyc",
     "condition": lambda ctx: ctx["service_type"] != "mobile"},
    {"field": "date_of_birth",                  "stage": "kyc"},
    {"field": "gender",                         "stage": "kyc"},
    {"field": "home_address",                   "stage": "kyc"},
    {"field": "identification_document_type",  "stage": "identity_verification"},
    {"field": "identification_document_number","stage": "identity_verification"},
    {"field": "subscriber_type",                "stage": "kyc"},
    {"field": "contract_type",                  "stage": "service_selection"},
    {"field": "payment_method",                 "stage": "service_selection"},
    {"field": "paperless_billing_preference",   "stage": "service_selection"},
    {"field": "phone_service_selection",        "stage": "service_selection",
     "condition": lambda ctx: ctx["service_type"] in ("mobile", "landline")},
    {"field": "internet_service_selection",     "stage": "service_selection",
     "condition": lambda ctx: ctx["service_type"] in ("broadband", "tv")},
    {"field": "streaming_addons_selection",     "stage": "service_selection",
     "condition": lambda ctx: ctx["service_type"] == "tv"},
    {"field": "subscriber_number",              "stage": "activation"},
]

TELECOM_STAGE_ORDER = [
    "kyc",
    "identity_verification",
    "service_selection",
    "activation",
]

FIELD_DEFINITIONS = {"banking": BANKING_FIELDS, "telecom": TELECOM_FIELDS}

STAGE_ORDER = {"banking": BANKING_STAGE_ORDER, "telecom": TELECOM_STAGE_ORDER}

SCENARIOS = ["normal_completion", "validation_errors", "revisited_fields",
             "long_completion_times", "incomplete_sessions"]

SCENARIO_WEIGHTS = {"normal_completion": 0.50, "validation_errors": 0.15,
                     "revisited_fields": 0.15, "long_completion_times": 0.10,
                     "incomplete_sessions": 0.10}

MAX_VALIDATION_RETRIES = 2          # a field can fail at most this many times before it passes

VALIDATION_FAILURE_PROBABILITY = 0.18

UNRESOLVABLE_REVISIT_PROBABILITY = 0.04


In [ ]:
def generate_customer_id(rng=random):
    return f"CUST{rng.randint(10000, 99999)}"

def generate_sales_agent_id(rng=random):
    return f"AGENT{rng.randint(100, 999)}"

def generate_session_id(rng=random):
    return f"SESSION-{rng.getrandbits(32):08X}"

def generate_session_context(industry, rng=random):
    if industry == "banking":
        customer_type = rng.choices(["individual", "business"], weights=[80, 20], k=1)[0]
        return {"customer_type": customer_type}
    if industry == "telecom":
        service_type = rng.choices(["mobile", "broadband", "tv", "iot", "landline"],
                                   weights=[35, 25, 20, 10, 10], k=1)[0]
        return {"service_type": service_type}
    raise ValueError("Industry must be 'banking' or 'telecom'")

def context_label(industry, context):
    """A single string identifying the session's context branch -- this is
    the explicit feature exposed to the model, known from the start of the
    session (not derived from anything in the future)."""
    if industry == "banking":
        return context["customer_type"]
    return context["service_type"]

def select_applicable_fields(industry, context, rng=random):
    """
    This session's target field list: every field whose condition (if any)
    passes and whose probability (if any) is met, ordered by the industry's
    FIXED stage order and FIXED within-stage order (definition order) --
    no shuffling. This is the main change from v1: a consistent script
    order is what makes "next field" learnable.
    """
    definitions = FIELD_DEFINITIONS[industry]
    stage_order = STAGE_ORDER[industry]

    selected = []
    for definition in definitions:
        condition = definition.get("condition")
        if condition is not None and not condition(context):
            continue
        probability = definition.get("probability", 1.0)
        if rng.random() > probability:
            continue
        selected.append(definition["field"])

    by_stage = {stage: [] for stage in stage_order}
    field_to_stage = {d["field"]: d["stage"] for d in definitions}
    for field in selected:
        by_stage[field_to_stage[field]].append(field)

    ordered_fields = []
    stage_of_field = {}
    for stage in stage_order:
        # NOTE: no random.shuffle() here -- definition order is preserved.
        for field in by_stage[stage]:
            stage_of_field[field] = stage
        ordered_fields.extend(by_stage[stage])

    return ordered_fields, stage_of_field

def select_scenario(rng=random):
    return rng.choices(list(SCENARIO_WEIGHTS.keys()),
                       weights=list(SCENARIO_WEIGHTS.values()), k=1)[0]

def generate_event_duration(scenario, rng=random):
    if scenario == "long_completion_times":
        return round(rng.uniform(15, 60), 2)
    if scenario == "validation_errors":
        return round(rng.uniform(5, 20), 2)
    if scenario == "revisited_fields":
        return round(rng.uniform(4, 18), 2)
    return round(rng.uniform(1, 12), 2)

def generate_failure_reason(rng=random):
    return rng.choice(["Required information missing", "Invalid information entered",
                       "Information could not be verified"])

def generate_agent_behaviour(validation_status, is_revisit, scenario, rng=random):
    if validation_status == "failed":
        action = rng.choices(
            ["corrected_information", "requested_missing_information", "reviewed_information"],
            weights=[50, 35, 15], k=1)[0]
        return (action, True, rng.randint(1, 2),
                rng.choices(["none", "required"], weights=[85, 15], k=1)[0])

    if is_revisit:
        action = rng.choices(
            ["revisited_information", "reviewed_information", "provided_guidance"],
            weights=[50, 35, 15], k=1)[0]
        return (action, True, rng.choice([0, 1]), "none")

    if scenario == "incomplete_sessions":
        action = rng.choices(
            ["provided_guidance", "requested_missing_information", "reviewed_information"],
            weights=[45, 40, 15], k=1)[0]
        return (action, True, rng.choice([0, 1]),
                rng.choices(["none", "required"], weights=[90, 10], k=1)[0])

    if scenario == "long_completion_times":
        action = rng.choices(
            ["reviewed_information", "provided_guidance", "entered_information", "completed_step"],
            weights=[40, 35, 15, 10], k=1)[0]
        return (action, True, rng.choice([0, 1]), "none")

    action = rng.choices(
        ["entered_information", "completed_step", "provided_guidance", "reviewed_information"],
        weights=[40, 35, 15, 10], k=1)[0]
    guidance_given = action == "provided_guidance"
    return (action, guidance_given, 0, "none")

def calculate_onboarding_risk(validation_status, event_duration, revisit_count, session_progress):
    risk = 0.05
    if validation_status == "failed":
        risk += 0.25
    if event_duration > 15:
        risk += 0.15
    if revisit_count > 0:
        risk += 0.10 * revisit_count
    if session_progress < 50:
        risk += 0.10
    return min(round(risk, 2), 1.00)

def get_risk_level(risk_score):
    if risk_score < 0.30:
        return "Low"
    if risk_score < 0.70:
        return "Medium"
    return "High"


In [ ]:
def generate_session(industry, scenario=None, rng: Optional[random.Random] = None,
                     start_time=None, session_id=None):
    if rng is None:
        rng = random.Random()
    industry = industry.lower()
    if industry not in ("banking", "telecom"):
        raise ValueError("Industry must be 'banking' or 'telecom'")
    if scenario is None:
        scenario = select_scenario(rng)

    session_id = session_id or generate_session_id(rng)
    customer_id = generate_customer_id(rng)
    sales_agent_id = generate_sales_agent_id(rng)

    context = generate_session_context(industry, rng)
    context_value = context_label(industry, context)
    target_fields, stage_of_field = select_applicable_fields(industry, context, rng)

    if start_time is None:
        start_time = datetime.now()

    if scenario == "incomplete_sessions" and len(target_fields) > 1:
        number_of_fields = rng.randint(1, len(target_fields) - 1)
        base_fields = target_fields[:number_of_fields]
    else:
        base_fields = target_fields.copy()

    # ------------------------------------------------------------------
    # Build the actual step-by-step sequence, resolving validation
    # retries and revisits with explicit, rule-based logic rather than
    # free-floating randomness.
    # ------------------------------------------------------------------
    field_sequence = []       # every field actually shown, in order
    validation_status_seq = []  # validation_status recorded at that step
    is_revisit_seq = []

    target_field_count = max(len(target_fields), 1)
    revisit_count = 0

    for field in base_fields:
        # -------- low-rate, deliberately UNRESOLVABLE revisit --------
        # Only in "revisited_fields" scenario sessions, and only ~4% of
        # steps within them -- nothing in the row predicts this, by design.
        # This keeps a small, honest amount of irreducible noise in the
        # data instead of a purely deterministic (and therefore unrealistic)
        # generator.
        if (scenario == "revisited_fields" and len(field_sequence) > 0
                and rng.random() < UNRESOLVABLE_REVISIT_PROBABILITY):
            revisited_field = rng.choice(field_sequence)
            field_sequence.append(revisited_field)
            validation_status_seq.append("passed")
            is_revisit_seq.append(True)
            revisit_count += 1

        # -------- enter the actual field, with bounded validation retries --------
        attempt = 0
        while True:
            will_fail = (
                scenario == "validation_errors"
                and attempt < MAX_VALIDATION_RETRIES
                and rng.random() < VALIDATION_FAILURE_PROBABILITY
            )
            field_sequence.append(field)
            validation_status_seq.append("failed" if will_fail else "passed")
            is_revisit_seq.append(attempt > 0)
            if will_fail:
                attempt += 1
                continue
            break

    # ------------------------------------------------------------------
    # Emit events
    # ------------------------------------------------------------------
    events = []
    current_time = start_time

    for index, field in enumerate(field_sequence):
        validation_status = validation_status_seq[index]
        is_revisit = is_revisit_seq[index]
        failure_reason = generate_failure_reason(rng) if validation_status == "failed" else ""
        event_status = "failed" if validation_status == "failed" else "completed"

        duration = generate_event_duration(scenario, rng)

        previous_field = field_sequence[index - 1] if index > 0 else ""
        next_field = field_sequence[index + 1] if index + 1 < len(field_sequence) else ""

        progress = round((len(set(field_sequence[:index + 1])) / target_field_count) * 100, 2)
        risk_score = calculate_onboarding_risk(validation_status, duration, revisit_count, progress)
        risk_level = get_risk_level(risk_score)

        (agent_action, agent_guidance_given, agent_correction_count,
         agent_escalation) = generate_agent_behaviour(validation_status, is_revisit,
                                                       scenario, rng)

        if agent_action in ["corrected_information", "requested_missing_information",
                             "reviewed_information", "revisited_information"]:
            agent_response_time = round(rng.uniform(1, 8), 2)
        elif agent_action == "provided_guidance":
            agent_response_time = round(rng.uniform(0.5, 5), 2)
        else:
            agent_response_time = round(rng.uniform(0.2, 3), 2)

        if agent_guidance_given:
            guidance = (f"Review the current information and proceed to {next_field}."
                        if next_field else
                        "Review the completed information and complete Customer Onboarding.")
        else:
            guidance = ""

        if scenario == "incomplete_sessions":
            session_status = "in_progress"
        elif index == len(field_sequence) - 1 and event_status == "completed":
            session_status = "completed"
        else:
            session_status = "in_progress"

        event_name = "revisited" if is_revisit else "entered"

        event = {
            "session_id": session_id, "customer_id": customer_id, "sales_agent_id": sales_agent_id,
            "industry": industry, "context_value": context_value,
            "onboarding_scenario": scenario,
            "onboarding_stage": stage_of_field[field], "onboarding_field": field,
            "event_name": event_name, "event_timestamp": current_time.isoformat(),
            "event_status": event_status, "event_duration_minutes": duration,
            "is_revisit": is_revisit, "previous_onboarding_field": previous_field,
            "next_onboarding_field": next_field, "validation_status": validation_status,
            "failure_reason": failure_reason, "onboarding_progress": progress,
            "onboarding_risk_score": risk_score, "onboarding_risk_level": risk_level,
            "agent_action": agent_action, "agent_response_time_minutes": agent_response_time,
            "agent_guidance_given": agent_guidance_given, "agent_correction_count": agent_correction_count,
            "agent_escalation": agent_escalation, "real_time_guidance": guidance,
            "session_status": session_status,
        }
        events.append(event)
        current_time += timedelta(minutes=duration)

    return events


In [ ]:
def _split_counts(group_count, ratios):
    raw_counts = [group_count * ratio for ratio in ratios.values()]
    counts = [int(count) for count in raw_counts]
    remaining = group_count - sum(counts)
    order = sorted(range(len(raw_counts)),
                   key=lambda index: raw_counts[index] - counts[index],
                   reverse=True)
    for index in order[:remaining]:
        counts[index] += 1
    return counts

def _assign_group_splits(events, group_column, split_column, rng, ratios,
                         stratify_columns=()):
    strata = defaultdict(lambda: defaultdict(list))
    for event in events:
        stratum = tuple(event[column] for column in stratify_columns)
        strata[stratum][event[group_column]].append(event)

    for groups in strata.values():
        group_ids = sorted(groups)
        rng.shuffle(group_ids)
        counts = _split_counts(len(group_ids), ratios)
        offset = 0
        for split_name, count in zip(ratios, counts):
            for group_id in group_ids[offset:offset + count]:
                for event in groups[group_id]:
                    event[split_column] = split_name
            offset += count

def assign_dataset_splits(events, seed, ratios=None):
    if ratios is None:
        ratios = {"train": 0.70, "validation": 0.15, "test": 0.15}
    if not ratios or any(not 0 < value <= 1 for value in ratios.values()):
        raise ValueError("Split ratios must be between zero and one")
    if abs(sum(ratios.values()) - 1.0) > 1e-9:
        raise ValueError("Split ratios must sum to 1")
    if not events:
        return events

    _assign_group_splits(
        events, "session_id", "split", random.Random(seed),
        ratios, ("industry", "onboarding_scenario"))
    _assign_group_splits(
        events, "sales_agent_id", "agent_split",
        random.Random(seed ^ 0x5F3759DF), ratios)
    return events

def generate_dataset(number_of_sessions=1000, industry="banking", seed=42,
                     split_ratios=None):
    industry = industry.lower()
    if number_of_sessions <= 0:
        raise ValueError("number_of_sessions must be greater than zero")
    if industry not in ("banking", "telecom", "both"):
        raise ValueError("Industry must be 'banking', 'telecom', or 'both'")

    rng = random.Random(seed)
    all_events = []
    start_date = datetime(2026, 1, 1)
    for index in range(number_of_sessions):
        if industry == "both":
            session_industry = "banking" if index % 2 == 0 else "telecom"
        else:
            session_industry = industry
        session_start = (start_date + timedelta(minutes=30 * index)
                         if seed is not None else None)
        all_events.extend(generate_session(
            session_industry,
            rng=rng,
            start_time=session_start,
            session_id=f"SESSION-{index + 1:08X}"))
    split_seed = seed if seed is not None else rng.getrandbits(32)
    return assign_dataset_splits(all_events, split_seed, split_ratios)

def save_to_csv(events, filename):
    if not events:
        return
    fieldnames = list(events[0].keys())
    output_path = Path(filename)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    with output_path.open("w", newline="", encoding="utf-8") as file:
        writer = csv.DictWriter(file, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(events)


## 3. Generate and save the CSV
The output path's parent directory is created automatically. Choose a new filename if you want to preserve an existing generated CSV.

In [ ]:
events = generate_dataset(
    number_of_sessions=NUMBER_OF_SESSIONS,
    industry=INDUSTRY,
    seed=SEED,
)
save_to_csv(events, OUTPUT_CSV)

print(f"Generated {len(events):,} events from {NUMBER_OF_SESSIONS:,} sessions")
print(f"Industry selection: {INDUSTRY}")
print(f"Seed: {SEED}")
print(f"Saved CSV: {OUTPUT_CSV.resolve()}")


## 4. Check the generated dataset
Run these assertions to verify group assignments and inspect the split sizes before training.

In [ ]:
from collections import Counter, defaultdict

session_splits = defaultdict(set)
agent_splits = defaultdict(set)
for event in events:
    session_splits[event["session_id"]].add(event["split"])
    agent_splits[event["sales_agent_id"]].add(event["agent_split"])

assert all(len(splits) == 1 for splits in session_splits.values()), "A session crossed splits"
assert all(len(splits) == 1 for splits in agent_splits.values()), "An agent crossed agent_split"

print("Events:", f"{len(events):,}")
print("Sessions:", f"{len(session_splits):,}")
print("Session-level split events:", dict(Counter(event["split"] for event in events)))
print("Agent-level split events:", dict(Counter(event["agent_split"] for event in events)))
print("Industries:", dict(Counter(event["industry"] for event in events)))
print("Sample event:", events[0])
print("Validation passed: no session or agent crosses its corresponding split.")


## Optional: download the CSV from Colab
Run this cell only in Google Colab if you want a browser download. It is skipped automatically in local Jupyter.

In [ ]:
try:
    from google.colab import files
except ImportError:
    print("Not running in Google Colab; CSV is available at the output path above.")
else:
    files.download(str(OUTPUT_CSV))
